In [5]:
import csv
import os
import random
import pickle
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import sklearn
import warnings
import skbold
from skbold.preproc import ConfoundRegressor
from scipy.stats import pearsonr
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [15]:
# Define ConfoundRegressor: skbold
def confound_regressor_skbold(features_train, features_test, confounds_train, confounds_test):
    # Scale features (train and test sets)
    scaler_features = StandardScaler()
    features_train_scaled = scaler_features.fit_transform(features_train)
    features_test_scaled = scaler_features.transform(features_test)
    
    # Scale confounds (train and test sets)
    scaler_confounds = StandardScaler()
    confounds_train_scaled = scaler_confounds.fit_transform(confounds_train)
    confounds_test_scaled = scaler_confounds.transform(confounds_test)

    # Convert full sets into np.array
    features_full_scaled_np = np.array(pd.concat([pd.DataFrame(features_train_scaled, columns = features_train.columns), pd.DataFrame(features_test_scaled, columns = features_test.columns)], axis=0))
    confounds_full_scaled_np = np.array(pd.concat([pd.DataFrame(confounds_train_scaled, columns = confounds_train.columns), pd.DataFrame(confounds_test_scaled, columns = confounds_test.columns)], axis=0))
    
    # Define ConfoundRegressor on a FULL set (train and test)
    cfr = ConfoundRegressor(confound=confounds_full_scaled_np, X=features_full_scaled_np)
    features_train_corrected = cfr.fit_transform(features_train_scaled)
    features_test_corrected = cfr.transform(features_test_scaled)


    return features_train_corrected, features_test_corrected, features_train_scaled, features_test_scaled, scaler_features

In [7]:
amplitudes_21 = pd.read_csv("/media/hcs-sci-psy-narun/IBu/UK_BB/ukbbdata/Resting_State/rsfMRI_matrices/amplitudes_tables/21/amplitudes_21_fin_CLEAN.csv")
amplitudes_55 = pd.read_csv("/media/hcs-sci-psy-narun/IBu/UK_BB/ukbbdata/Resting_State/rsfMRI_matrices/amplitudes_tables/55/amplitudes_55_fin_CLEAN.csv")
full_correlation_21 = pd.read_csv('/media/hcs-sci-psy-narun/IBu/UK_BB/ukbbdata/Resting_State/rsfMRI_matrices/full_corr_tables/21/fcorr_21_instance_2_FINAL.csv')
full_correlation_55 = pd.read_csv('/media/hcs-sci-psy-narun/IBu/UK_BB/ukbbdata/Resting_State/rsfMRI_matrices/full_corr_tables/55/fcorr_55_instance_2_FINAL.csv')
partial_correlation_21 = pd.read_csv("/media/hcs-sci-psy-narun/IBu/UK_BB/ukbbdata/Resting_State/rsfMRI_matrices/partial_corr_tables/21/pcorr_21_instance_2_FINAL.csv")
partial_correlation_55 = pd.read_csv("/media/hcs-sci-psy-narun/IBu/UK_BB/ukbbdata/Resting_State/rsfMRI_matrices/partial_corr_tables/55/pcorr_55_instance_2_FINAL.csv")

Remember that renamed files are here: '/media/hcs-sci-psy-narun/IBu/UK_BB/ukbbdata/Resting_State/rsfMRI_matrices/amplitudes_tables/21/{df}_fin_renamed'

In [8]:
# Define modalities
modalities = [
'amplitudes_21',
'amplitudes_55',
'full_correlation_21',
'full_correlation_55',
'partial_correlation_21',
'partial_correlation_55']

In [18]:

confounds = pd.read_csv('/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/resting_files/rs_confounds.csv')

# these are the same confounds as at '/media/hcs-sci-psy-narun/IBu/UK_BB/ML_DATASETS/Brain/rsMRI/rs_confounds_fin_full.csv' saved at ICA_Confounds step

warnings.simplefilter(action='ignore', category=FutureWarning)

############## 1
seed = 42

for modality in modalities:

    print(f'Started {modality}', flush=True)

    file = pd.read_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/resting_files/{modality}.csv').rename(columns={'ID': 'eid'})

    folds = ["0", "1", "2", "3", "4"]
    pls_result = {}
    
    # Match confounds to MRI
    print(f'Matching brain data to confounds in {modality}', flush=True)

    conf_to_brain_match = pd.merge(confounds, file['eid'], on='eid')
    brain_to_conf_match = pd.merge(conf_to_brain_match['eid'], file, on='eid')

    for fold in folds:

        train_id = pd.read_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/g_factor_5_folds_python/fold_{fold}/train_id_fold_{fold}.csv')
        test_id = pd.read_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/g_factor_5_folds_python/fold_{fold}/test_id_fold_{fold}.csv')
        
        # Upload g-factor with ID
        g_train_full = pd.read_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/g_factor/g_train_with_id_fold_{fold}.csv')
        g_test_full = pd.read_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/g_factor/g_test_with_id_fold_{fold}.csv')

    
        # Match brain data to cognitive data
        print(f'Matching brain data to confounds in {modality}', flush=True)
        brain_train, brain_test, brain_train_id, brain_test_id = pd.merge(brain_to_conf_match, train_id, on='eid').drop(columns=['eid']), pd.merge(brain_to_conf_match, test_id, on='eid').drop(columns=['eid']), pd.merge(brain_to_conf_match, train_id, on='eid')['eid'], pd.merge(brain_to_conf_match, test_id, on='eid')['eid']

        brain_train.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/{modality}_train_fold_{fold}.csv', index=False)
        brain_test.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/{modality}_test_fold_{fold}.csv', index=False)
        brain_train_id.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/{modality}_train_id_fold_{fold}.csv', index=False)
        brain_test_id.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/{modality}_test_id_fold_{fold}.csv', index=False)
        
        ############## 2
        print(f'Matching confounds to {modality} fold {fold}', flush=True)
        
        # Match confounds to MRI
        brain_conf_train, brain_conf_test = pd.merge(conf_to_brain_match, brain_train_id, on='eid').drop(columns=['eid']), pd.merge(conf_to_brain_match, brain_test_id, on='eid').drop(columns=['eid'])
        brain_conf_train.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/{modality}_conf_train_fold_{fold}.csv', index=False)
        brain_conf_test.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/{modality}_conf_test_fold_{fold}.csv', index=False)
        
        ############## 3
        print(f'Matching g-factor to {modality} fold {fold}', flush=True)
        
        # Match g-factor back to MRI
        g_train, g_test, g_train_id, g_test_id = pd.merge(g_train_full, brain_train_id, on='eid').drop(columns=['eid']), pd.merge(g_test_full, brain_test_id, on='eid').drop(columns=['eid']), pd.merge(g_train_full, brain_train_id, on='eid')['eid'], pd.merge(g_test_full, brain_test_id, on='eid')['eid']
        g_train.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/g_train_{modality}_matched_fold_{fold}.csv', index=False)
        g_test.to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/suppl/g_test_{modality}_matched_fold_{fold}.csv', index=False)
        
        ############## 4
        print(f'Applying ConfoundRegressor to {modality} fold {fold}', flush=True)
        
        # Apply ConfoundRegressor
        features_train_corr, features_test_corr, features_train_scaled, features_test_scaled, scaler_features = confound_regressor_skbold(brain_train, brain_test, brain_conf_train, brain_conf_test)
        pd.DataFrame(features_train_corr, columns = brain_train.columns).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/scaling/{modality}_train_corr_{fold}.csv', index=False)
        pd.DataFrame(features_test_corr, columns = brain_test.columns).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/scaling/{modality}_test_corr_{fold}.csv', index=False)
        
        pd.DataFrame(features_train_scaled, columns = brain_train.columns).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/scaling/{modality}_train_scaled_{fold}.csv', index=False)
        pd.DataFrame(features_test_scaled, columns = brain_test.columns).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/scaling/{modality}_test_scaled_{fold}.csv', index=False)
        
        with open(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/scaling/scaler_features_{modality}_fold_{fold}.pkl', "wb") as f:
            pickle.dump(scaler_features, f)

        # Initiate and run PLS
        parameters = {'n_components': range(1, 36, 1)}
        pls = PLSRegression()
        model = GridSearchCV(pls, parameters, scoring = 'neg_mean_absolute_error', cv=KFold(10, shuffle = True, random_state=seed), verbose=4, n_jobs = 15)
        
        
        print(f'Fitting PLS to {modality} fold {fold}', flush=True)
        model.fit(features_train_corr, np.array(g_train))
        
        print(f'Model parameters for fold {fold}:', model.cv_results_['params'])
        print(f'Mean test score for fold {fold}:', model.cv_results_['mean_test_score'])
        print(f'Rank test score for fold {fold}:', model.cv_results_['rank_test_score'])
        print(model)
        
        print(f'Saving PLS model for {modality} fold {fold}')
        with open(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/models/pkl/{modality}_model_fold_{fold}.pkl', "wb") as f:
            pickle.dump(model, f)
            
        print(f'Best params in fold {fold} = ', model.best_params_)
        print(f'Best score (neg_mean_absolute_error) in fold {fold} = ', model.best_score_)
            
        # Predict the values
        print(f'Predicting & saving g_test for {modality} fold {fold}', flush=True)
        g_pred_test = model.predict(np.array(features_test_corr))
        pd.DataFrame(g_pred_test, columns=['g predicted test']).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/g_pred/{modality}_test_fold_{fold}.csv')

        g_pred_test_with_id = pd.concat([g_test_id.astype(int), pd.DataFrame(g_pred_test, columns=['g predicted test'])], axis=1).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/g_pred/{modality}_g_pred_test_id_fold_{fold}.csv')

        
        print(f'Predicting & saving g_train for {modality} fold {fold}', flush=True)
        g_pred_train = model.predict(np.array(features_train_corr))
        pd.DataFrame(g_pred_train, columns=['g predicted train']).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/g_pred/{modality}_g_pred_train_fold_{fold}.csv')
        

        g_pred_train_with_id = pd.concat([g_train_id.astype(int), pd.DataFrame(g_pred_train, columns=['g predicted train'])], axis=1).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/g_pred/{modality}_g_pred_train_id_fold_{fold}.csv')
        
            
        print(f"Fold = {fold}")
        print("----------")
        print("MSE = ", mean_squared_error(np.array(g_test)[:,0], g_pred_test[:,0]))
        print("MAE = ", mean_absolute_error(np.array(g_test)[:,0], g_pred_test[:,0]))
        print("R2 = ", r2_score(np.array(g_test)[:,0], g_pred_test[:,0]))
        print("Pearson's r = ", pearsonr(np.array(g_test)[:,0], g_pred_test[:,0]))
        print("----------")
            
        pls_result['fold'] = fold
        pls_result['modality'] = modality
        pls_result['n_components'] = model.best_params_
        pls_result['MSE'] = mean_squared_error(np.array(g_test)[:,0], g_pred_test[:,0])
        pls_result['MAE'] = mean_absolute_error(np.array(g_test)[:,0], g_pred_test[:,0])
        pls_result['R2'] = r2_score(np.array(g_test)[:,0], g_pred_test[:,0])
        pls_result['Pearson r'] = pearsonr(np.array(g_test)[:,0], g_pred_test[:,0])
            
        with open(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/models/csv/{modality}_fold_{fold}_PLS_result.csv', 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=pls_result.keys())
            writer.writerow(pls_result)
            
        pls_result.clear()
        
        corr, pval = stats.pearsonr(np.squeeze(np.array(g_test)), np.squeeze(g_pred_test))
        r2 = r2_score(np.squeeze(np.array(g_test)), np.squeeze(g_pred_test))
        mse = mean_squared_error(np.squeeze(np.array(g_test)), np.squeeze(g_pred_test))
        result = pd.DataFrame([modality, fold, corr, pval, r2, mse, model.best_params_], index=['Modality', 'Fold', 'Correlation', 'P-value', 'R2', 'MSE', 'n components'], columns=['Values']).to_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/models/csv/{modality}_fold_{fold}_full_result.csv')

Started amplitudes_21
Matching brain data to confounds in amplitudes_21
Matching brain data to confounds in amplitudes_21
Matching confounds to amplitudes_21 fold 0
Matching g-factor to amplitudes_21 fold 0
Applying ConfoundRegressor to amplitudes_21 fold 0
Fitting PLS to amplitudes_21 fold 0
Fitting 10 folds for each of 35 candidates, totalling 350 fits
[CV 8/10] END ..................n_components=1;, score=-0.568 total time=   0.1s
[CV 7/10] END ..................n_components=1;, score=-0.560 total time=   0.1s[CV 9/10] END ..................n_components=1;, score=-0.541 total time=   0.1s

[CV 1/10] END ..................n_components=2;, score=-0.563 total time=   0.1s
[CV 1/10] END ..................n_components=1;, score=-0.565 total time=   0.1s
[CV 6/10] END ..................n_components=1;, score=-0.562 total time=   0.1s
[CV 5/10] END ..................n_components=1;, score=-0.568 total time=   0.1s
[CV 4/10] END ..................n_components=1;, score=-0.562 total time=   

/home/irinabu/anaconda3/envs/ukbiobank_py11/lib/python3.11/site-packages/sklearn/model_selection/_search.py:976: UserWarning: One or more of the test scores are non-finite: [-0.56322142 -0.56100257 -0.56068459 -0.56067507 -0.56065977 -0.56070898
 -0.56070173 -0.56070281 -0.56070175 -0.56070179 -0.56070182 -0.56070182
 -0.56070182 -0.56070182 -0.56070182 -0.56070182 -0.56070182 -0.56070182
 -0.56070182 -0.56070182 -0.56070182 -0.56070182         nan         nan
         nan         nan         nan         nan         nan         nan
         nan         nan         nan         nan         nan]
  warnings.warn(


Fold = 0
----------
MSE =  0.49278475366655955
MAE =  0.5605049371250163
R2 =  0.019600265251767945
Pearson's r =  PearsonRResult(statistic=0.14096037133418854, pvalue=5.843923819802905e-25)
----------
Matching brain data to confounds in amplitudes_21
Matching confounds to amplitudes_21 fold 1
Matching g-factor to amplitudes_21 fold 1
Applying ConfoundRegressor to amplitudes_21 fold 1
Fitting PLS to amplitudes_21 fold 1
Fitting 10 folds for each of 35 candidates, totalling 350 fits
[CV 1/10] END ..................n_components=1;, score=-0.661 total time=   0.0s
[CV 6/10] END ..................n_components=1;, score=-0.664 total time=   0.0s
[CV 2/10] END ..................n_components=1;, score=-0.668 total time=   0.0s
[CV 5/10] END ..................n_components=1;, score=-0.665 total time=   0.0s
[CV 7/10] END ..................n_components=1;, score=-0.679 total time=   0.0s
[CV 8/10] END ..................n_components=1;, score=-0.662 total time=   0.0s
[CV 3/10] END .............

/home/irinabu/anaconda3/envs/ukbiobank_py11/lib/python3.11/site-packages/sklearn/model_selection/_search.py:976: UserWarning: One or more of the test scores are non-finite: [-0.66633082 -0.66530553 -0.66503037 -0.66505418 -0.66507967 -0.66511321
 -0.66511251 -0.66510495 -0.66510336 -0.66510319 -0.66510317 -0.66510316
 -0.66510316 -0.66510316 -0.66510316 -0.66510316 -0.66510316 -0.66510316
 -0.66510316 -0.66510316 -0.66510316 -0.66510316         nan         nan
         nan         nan         nan         nan         nan         nan
         nan         nan         nan         nan         nan]
  warnings.warn(


Fold = 1
----------
MSE =  0.6876849658433719
MAE =  0.6606863302671802
R2 =  0.0023987943096756004
Pearson's r =  PearsonRResult(statistic=0.0645637286985096, pvalue=2.505758331891441e-06)
----------
Matching brain data to confounds in amplitudes_21
Matching confounds to amplitudes_21 fold 2
Matching g-factor to amplitudes_21 fold 2
Applying ConfoundRegressor to amplitudes_21 fold 2
Fitting PLS to amplitudes_21 fold 2
Fitting 10 folds for each of 35 candidates, totalling 350 fits
[CV 3/10] END ..................n_components=1;, score=-0.675 total time=   0.0s
[CV 4/10] END ..................n_components=1;, score=-0.692 total time=   0.0s
[CV 7/10] END ..................n_components=1;, score=-0.699 total time=   0.0s
[CV 1/10] END ..................n_components=1;, score=-0.681 total time=   0.0s
[CV 6/10] END ..................n_components=1;, score=-0.688 total time=   0.0s
[CV 2/10] END ..................n_components=1;, score=-0.694 total time=   0.0s
[CV 8/10] END ..............

...

Fold = 4
----------
MSE =  0.5153165510159669
MAE =  0.5643535999245028
R2 =  0.018271613342813242
Pearson's r =  PearsonRResult(statistic=0.1377301113689935, pvalue=4.4180304939263486e-24)
----------
Started amplitudes_55
Matching brain data to confounds in amplitudes_55
Matching brain data to confounds in amplitudes_55
Matching confounds to amplitudes_55 fold 0
Matching g-factor to amplitudes_55 fold 0
Applying ConfoundRegressor to amplitudes_55 fold 0
Fitting PLS to amplitudes_55 fold 0
Fitting 10 folds for each of 35 candidates, totalling 350 fits
[CV 6/10] END ..................n_components=1;, score=-0.560 total time=   0.1s
[CV 1/10] END ..................n_components=1;, score=-0.565 total time=   0.1s
[CV 3/10] END ..................n_components=1;, score=-0.577 total time=   0.1s
[CV 2/10] END ..................n_components=1;, score=-0.556 total time=   0.1s
[CV 8/10] END ..................n_components=1;, score=-0.569 total time=   0.1s
[CV 2/10] END ..................n_com

/home/irinabu/anaconda3/envs/ukbiobank_py11/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:319: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(Yk, y_weights) / y_ss


[CV 9/10] END .................n_components=33;, score=-0.565 total time=   0.7s
[CV 5/10] END .................n_components=34;, score=-0.558 total time=   0.9s
[CV 8/10] END .................n_components=30;, score=-0.581 total time=   0.8s
[CV 6/10] END .................n_components=33;, score=-0.560 total time=   0.5s
[CV 10/10] END ................n_components=31;, score=-0.557 total time=   0.8s
[CV 7/10] END .................n_components=34;, score=-0.577 total time=   0.9s
[CV 10/10] END ................n_components=34;, score=-0.557 total time=   0.6s
[CV 9/10] END .................n_components=34;, score=-0.565 total time=   0.8s
[CV 10/10] END ................n_components=33;, score=-0.557 total time=   0.4s
[CV 8/10] END .................n_components=35;, score=-0.581 total time=   0.3s
[CV 2/10] END .................n_components=33;, score=-0.565 total time=   0.5s
[CV 4/10] END .................n_components=35;, score=-0.571 total time=   0.7s
[CV 2/10] END ..............

/home/irinabu/anaconda3/envs/ukbiobank_py11/lib/python3.11/site-packages/sklearn/cross_decomposition/_pls.py:319: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(Yk, y_weights) / y_ss


Predicting & saving g_train for amplitudes_55 fold 3
Fold = 3
----------
MSE =  0.4942374357027007
MAE =  0.5575324704073772
R2 =  0.02935349441119317
Pearson's r =  PearsonRResult(statistic=0.17152486651424337, pvalue=1.426746137663854e-36)
----------
Matching brain data to confounds in amplitudes_55
Matching confounds to amplitudes_55 fold 4
Matching g-factor to amplitudes_55 fold 4
Applying ConfoundRegressor to amplitudes_55 fold 4
Fitting PLS to amplitudes_55 fold 4
Fitting 10 folds for each of 35 candidates, totalling 350 fits
[CV 5/10] END ..................n_components=1;, score=-0.566 total time=   0.0s
[CV 2/10] END ..................n_components=1;, score=-0.586 total time=   0.1s
[CV 3/10] END ..................n_components=1;, score=-0.571 total time=   0.1s
[CV 10/10] END .................n_components=1;, score=-0.558 total time=   0.1s
[CV 9/10] END ..................n_components=1;, score=-0.568 total time=   0.1s[CV 4/10] END ..................n_components=2;, score=-0.

In [23]:
five_folds = []
folds = ["0", "1", "2", "3", "4"]
for modality in modalities:
    for fold in folds:
        pls = pd.read_csv(f'/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/rs/ica_main/fold_{fold}/models/csv/{modality}_fold_{fold}_PLS_result.csv', header=None)
        pls.columns = ['Fold', 'Modality', 'n components', 'MSE', 'MAE', 'R2', 'Pearson r']
        #pls.index = [modality] * len(pls)
        five_folds.append(pls)
        five_folds_all_modalities = pd.concat(five_folds, ignore_index=False)

five_folds_all_modalities['Pearson r'] = five_folds_all_modalities['Pearson r'].astype(str).str.replace(r'PearsonRResult\(statistic=|pvalue=|\)', '', regex=True)
five_folds_all_modalities[['Pearson r', 'p-value']] = five_folds_all_modalities['Pearson r'].str.split(',', expand=True).astype(float).round(decimals=3)
five_folds_all_modalities = five_folds_all_modalities.round(decimals=3)
#five_folds_all_modalities.to_csv('/media/hcs-sci-psy-narun/IBu/UK_BB/Cog-Ment/PLS/brain/dti/pls_5_folds_all_modalities.csv', index=False)
with pd.option_context('display.max_rows', None):
    display(five_folds_all_modalities)

,Fold,Modality,n components,MSE,MAE,R2,Pearson r,p-value
0,0,amplitudes_21,{'n_components': 5},0.493,0.561,0.020,0.141,0.0
0,1,amplitudes_21,{'n_components': 3},0.688,0.661,0.002,0.065,0.0
0,2,amplitudes_21,{'n_components': 5},0.776,0.706,0.007,0.085,0.0
0,3,amplitudes_21,{'n_components': 3},0.500,0.561,0.017,0.132,0.0
0,4,amplitudes_21,{'n_components': 5},0.515,0.564,0.018,0.138,0.0
0,0,amplitudes_55,{'n_components': 5},0.488,0.558,0.028,0.169,0.0
0,1,amplitudes_55,{'n_components': 3},0.683,0.659,0.009,0.100,0.0
0,2,amplitudes_55,{'n_components': 3},0.770,0.701,0.014,0.121,0.0
0,3,amplitudes_55,{'n_components': 4},0.494,0.558,0.029,0.172,0.0
0,4,amplitudes_55,{'n_components': 3},0.516,0.566,0.017,0.136,0.0


In [24]:
# Average across folds
five_folds_all_modalities_mean = five_folds_all_modalities[['R2', 'Pearson r', 'Modality', 'MSE', 'MAE']]
five_folds_all_modalities_mean.groupby(['Modality']).mean().round(3).sort_values(by='R2', ascending=False)

,R2,Pearson r,MSE,MAE
Modality,,,,
partial_correlation_55,0.076,0.282,0.558,0.591
full_correlation_55,0.058,0.245,0.569,0.596
partial_correlation_21,0.048,0.221,0.574,0.599
full_correlation_21,0.044,0.212,0.576,0.601
amplitudes_55,0.019,0.140,0.590,0.608
amplitudes_21,0.013,0.112,0.594,0.611
